In [9]:
"""
evaluate_synthetic_data.py
============================
Script evaluasi kualitas data sintetis tabular untuk kebutuhan kompetisi
data mining. Mengecek 4 dimensi utama:

1. FIDELITY  - seberapa mirip distribusi statistik data sintetis vs asli
2. UTILITY   - apakah data sintetis membawa "sinyal" yang sama untuk model ML
               (metode Train-Synthetic-Test-Real / TSTR)
3. PRIVACY   - apakah ada baris sintetis yang terlalu mirip/identik dengan
               data asli (Distance to Closest Record / DCR)
4. VISUAL    - proyeksi PCA 2D untuk sanity-check sebaran data

Cara pakai:
-----------
    pip install pandas numpy scikit-learn scipy matplotlib seaborn --break-system-packages

    python evaluate_synthetic_data.py \
        --real medical.csv \
        --synthetic medical.csv \
        --target nama_kolom_target \
        --task classification \
        --outdir hasil_evaluasi

Kalau tidak punya kolom target (misalnya baru cek kemiripan statistik saja),
--target boleh dikosongkan, nanti evaluasi utility (TSTR) otomatis dilewati.

Output:
-------
- Laporan ringkasan tercetak di terminal (skor akhir + rekomendasi lulus/tidak)
- File `laporan_evaluasi.txt` berisi detail lengkap
- File `perbandingan_distribusi.png` (histogram tiap kolom numerik)
- File `korelasi_asli_vs_sintetis.png` (heatmap korelasi)
- File `pca_asli_vs_sintetis.png` (sebaran 2D)
"""

'\nevaluate_synthetic_data.py\n============================\nScript evaluasi kualitas data sintetis tabular untuk kebutuhan kompetisi\ndata mining. Mengecek 4 dimensi utama:\n\n1. FIDELITY  - seberapa mirip distribusi statistik data sintetis vs asli\n2. UTILITY   - apakah data sintetis membawa "sinyal" yang sama untuk model ML\n               (metode Train-Synthetic-Test-Real / TSTR)\n3. PRIVACY   - apakah ada baris sintetis yang terlalu mirip/identik dengan\n               data asli (Distance to Closest Record / DCR)\n4. VISUAL    - proyeksi PCA 2D untuk sanity-check sebaran data\n\nCara pakai:\n-----------\n    pip install pandas numpy scikit-learn scipy matplotlib seaborn --break-system-packages\n\n    python evaluate_synthetic_data.py         --real medical.csv         --synthetic medical.csv         --target nama_kolom_target         --task classification         --outdir hasil_evaluasi\n\nKalau tidak punya kolom target (misalnya baru cek kemiripan statistik saja),\n--target boleh

In [10]:
import argparse
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # aman dijalankan tanpa display (server/headless)
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ks_2samp, chi2_contingency
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, r2_score, mean_absolute_error
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


In [11]:
# ----------------------------------------------------------------------
# 0. SINTESIS DATA (Scikit-Learn GMM)
# ----------------------------------------------------------------------
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
import numpy as np

def generate_synthetic_data(real_df: pd.DataFrame, epochs: int = 50) -> pd.DataFrame:
    """
    Menggunakan Gaussian Mixture Models (GMM) dari Scikit-Learn
    sebagai alternatif CTGAN untuk menghindari error Python 3.13 di Windows.
    (Parameter epochs diabaikan untuk GMM, menggunakan n_components=10)
    """
    print(f"\n=== 0. GENERATE SYNTHETIC DATA (GMM) ===")
    n_components = 10  # Jumlah komponen distribusi campuran GMM
    
    print(f"  Melatih model Gaussian Mixture (GMM) dengan {n_components} komponen...")
    
    # 1. Encoding & Scaling
    encoders = {}
    scalers = {}
    df_encoded = pd.DataFrame(index=real_df.index)
    
    for col in real_df.columns:
        if pd.api.types.is_numeric_dtype(real_df[col]):
            # Ubah distribusi fitur menjadi mendekati normal agar optimal untuk GMM
            scaler = QuantileTransformer(output_distribution='normal', random_state=42)
            df_encoded[col] = scaler.fit_transform(real_df[[col]]).flatten()
            scalers[col] = scaler
        else:
            # Encode variabel kategorikal
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(real_df[col].astype(str))
            encoders[col] = le
            
    # 2. Training Model GMM
    gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
    gmm.fit(df_encoded)
    
    # 3. Sampling/Mencetak Data Sintetis Baru
    print(f"  Men-generate {len(real_df)} baris data sintetis...")
    synthetic_encoded, _ = gmm.sample(len(real_df))
    synthetic_data = pd.DataFrame(synthetic_encoded, columns=real_df.columns)
    
    # 4. Inverse Transform (Mengembalikan data ke wujud asli)
    for col in real_df.columns:
        if pd.api.types.is_numeric_dtype(real_df[col]):
            synthetic_data[col] = scalers[col].inverse_transform(synthetic_data[[col]]).flatten()
        else:
            # GMM menghasilkan float, maka kita bulatkan untuk indeks klasifikasi
            synthetic_data[col] = np.round(synthetic_data[col]).astype(int)
            # Pastikan index tidak keluar dari batas
            max_class = len(encoders[col].classes_) - 1
            synthetic_data[col] = np.clip(synthetic_data[col], 0, max_class)
            # Kembalikan ke label teks semula
            synthetic_data[col] = encoders[col].inverse_transform(synthetic_data[col])
            
    print("  [SELESAI] Data sintetis berhasil dibuat!")
    return synthetic_data

In [12]:
# ----------------------------------------------------------------------
# UTIL
# ----------------------------------------------------------------------

def split_column_types(df: pd.DataFrame):
    """Pisahkan kolom numerik vs kategorikal."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    return numeric_cols, categorical_cols


def align_columns(real: pd.DataFrame, synth: pd.DataFrame):
    """Pastikan kedua dataframe punya kolom yang sama & urutan sama."""
    common_cols = [c for c in real.columns if c in synth.columns]
    missing_in_synth = set(real.columns) - set(synth.columns)
    missing_in_real = set(synth.columns) - set(real.columns)
    if missing_in_synth:
        print(f"[PERINGATAN] Kolom ada di data asli tapi tidak di sintetis: {missing_in_synth}")
    if missing_in_real:
        print(f"[PERINGATAN] Kolom ada di data sintetis tapi tidak di asli: {missing_in_real}")
    return real[common_cols].copy(), synth[common_cols].copy()

In [13]:
# ----------------------------------------------------------------------
# 1. FIDELITY
# ----------------------------------------------------------------------

def evaluate_fidelity(real: pd.DataFrame, synth: pd.DataFrame, outdir: Path):
    print("\n=== 1. FIDELITY (kemiripan statistik) ===")
    numeric_cols, categorical_cols = split_column_types(real)

    scores = []
    detail_lines = []

    # --- Numerik: Kolmogorov-Smirnov test ---
    for col in numeric_cols:
        r = real[col].dropna()
        s = synth[col].dropna()
        if len(r) == 0 or len(s) == 0:
            continue
        stat, pval = ks_2samp(r, s)
        # KS statistic mendekati 0 = mirip. Ubah jadi skor "kemiripan" 0-100
        similarity = max(0.0, 1 - stat) * 100
        scores.append(similarity)
        detail_lines.append(
            f"  [Numerik] {col:<25s} KS-stat={stat:.3f}  p-value={pval:.3f}  skor_mirip={similarity:.1f}"
        )

    # --- Kategorikal: Chi-square test terhadap distribusi frekuensi ---
    for col in categorical_cols:
        r_counts = real[col].value_counts()
        s_counts = synth[col].value_counts()
        categories = sorted(set(r_counts.index) | set(s_counts.index))
        r_freq = np.array([r_counts.get(c, 0) for c in categories]) + 1  # +1 laplace smoothing
        s_freq = np.array([s_counts.get(c, 0) for c in categories]) + 1
        # samakan skala total supaya chi2 valid
        s_freq_scaled = s_freq * (r_freq.sum() / s_freq.sum())
        try:
            stat, pval, _, _ = chi2_contingency(np.array([r_freq, s_freq_scaled]))
            similarity = max(0.0, 1 - min(stat / (r_freq.sum()), 1)) * 100
        except Exception:
            similarity = np.nan
            pval = np.nan
        scores.append(similarity if not np.isnan(similarity) else 0)
        detail_lines.append(
            f"  [Kategorikal] {col:<22s} chi2-stat_terskala  p-value={pval:.3f}  skor_mirip={similarity:.1f}"
        )

    fidelity_score = float(np.nanmean(scores)) if scores else 0.0
    print(f"  Skor fidelity rata-rata: {fidelity_score:.1f} / 100")
    for line in detail_lines:
        print(line)

    # --- Plot histogram per kolom numerik ---
    if numeric_cols:
        n = len(numeric_cols)
        ncols = 2
        nrows = n
        fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4 * nrows))
        if nrows == 1:
            axes = np.array([axes])
        
        for i, col in enumerate(numeric_cols):
            ax_real = axes[i, 0]
            ax_synth = axes[i, 1]
            
            sns.histplot(real[col].dropna(), color="steelblue", label="Asli", stat="density", alpha=0.5, ax=ax_real, kde=True)
            ax_real.set_title(f"{col} - Asli")
            ax_real.legend(fontsize=8)
            
            sns.histplot(synth[col].dropna(), color="orangered", label="Sintetis", stat="density", alpha=0.5, ax=ax_synth, kde=True)
            ax_synth.set_title(f"{col} - Sintetis")
            ax_synth.legend(fontsize=8)

        plt.tight_layout()
        plt.savefig(outdir / "perbandingan_distribusi.png", dpi=120)
        plt.close()
        print(f"  -> Plot distribusi disimpan: {outdir / 'perbandingan_distribusi.png'}")

    # --- Heatmap korelasi (numerik saja) ---
    if len(numeric_cols) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        sns.heatmap(real[numeric_cols].corr(), ax=axes[0], cmap="coolwarm", vmin=-1, vmax=1, annot=False)
        axes[0].set_title("Korelasi - Data Asli")
        sns.heatmap(synth[numeric_cols].corr(), ax=axes[1], cmap="coolwarm", vmin=-1, vmax=1, annot=False)
        axes[1].set_title("Korelasi - Data Sintetis")
        plt.tight_layout()
        plt.savefig(outdir / "korelasi_asli_vs_sintetis.png", dpi=120)
        plt.close()
        print(f"  -> Plot korelasi disimpan: {outdir / 'korelasi_asli_vs_sintetis.png'}")

    return fidelity_score, detail_lines

In [14]:
# ----------------------------------------------------------------------
# 2. UTILITY (Train-Synthetic-Test-Real)
# ----------------------------------------------------------------------

def evaluate_utility(real: pd.DataFrame, synth: pd.DataFrame, target: str, task: str):
    print("\n=== 2. UTILITY (Train-Synthetic-Test-Real) ===")

    if target not in real.columns or target not in synth.columns:
        print("  [DILEWATI] Kolom target tidak ditemukan di salah satu data.")
        return None

    df_real = real.dropna(subset=[target]).copy()
    df_synth = synth.dropna(subset=[target]).copy()

    feature_cols = [c for c in df_real.columns if c != target]

    # Encode kategorikal sederhana (label encoding gabungan asli+sintetis
    # supaya konsisten)
    combined = pd.concat([df_real[feature_cols], df_synth[feature_cols]], axis=0)
    encoders = {}
    for col in feature_cols:
        if not pd.api.types.is_numeric_dtype(combined[col]):
            le = LabelEncoder()
            combined[col] = combined[col].astype(str)
            le.fit(combined[col])
            encoders[col] = le

    def transform(df):
        out = df[feature_cols].copy()
        for col, le in encoders.items():
            out[col] = le.transform(out[col].astype(str))
        return out.fillna(out.median(numeric_only=True))

    X_real = transform(df_real)
    y_real = df_real[target]
    X_synth = transform(df_synth)
    y_synth = df_synth[target]

    if task == "classification" and not pd.api.types.is_numeric_dtype(y_real):
        y_le = LabelEncoder()
        y_le.fit(pd.concat([y_real.astype(str), y_synth.astype(str)]))
        y_real = y_le.transform(y_real.astype(str))
        y_synth = y_le.transform(y_synth.astype(str))

    # holdout dari data ASLI sebagai test set yang konsisten
    X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
        X_real, y_real, test_size=0.3, random_state=42
    )

    if task == "classification":
        model_baseline = RandomForestClassifier(n_estimators=200, random_state=42)
        model_tstr = RandomForestClassifier(n_estimators=200, random_state=42)
    else:
        model_baseline = RandomForestRegressor(n_estimators=200, random_state=42)
        model_tstr = RandomForestRegressor(n_estimators=200, random_state=42)

    # Baseline: latih & uji di data asli
    model_baseline.fit(X_train_real, y_train_real)
    pred_baseline = model_baseline.predict(X_test_real)

    # TSTR: latih di data sintetis, uji di data asli (holdout yang sama)
    model_tstr.fit(X_synth, y_synth)
    pred_tstr = model_tstr.predict(X_test_real)

    if task == "classification":
        score_baseline = accuracy_score(y_test_real, pred_baseline)
        score_tstr = accuracy_score(y_test_real, pred_tstr)
        f1_baseline = f1_score(y_test_real, pred_baseline, average="weighted")
        f1_tstr = f1_score(y_test_real, pred_tstr, average="weighted")
        print(f"  Baseline (train=asli, test=asli)   -> Accuracy={score_baseline:.3f}  F1={f1_baseline:.3f}")
        print(f"  TSTR     (train=sintetis, test=asli) -> Accuracy={score_tstr:.3f}  F1={f1_tstr:.3f}")
        metric_name = "Accuracy"
    else:
        score_baseline = r2_score(y_test_real, pred_baseline)
        score_tstr = r2_score(y_test_real, pred_tstr)
        mae_baseline = mean_absolute_error(y_test_real, pred_baseline)
        mae_tstr = mean_absolute_error(y_test_real, pred_tstr)
        print(f"  Baseline (train=asli, test=asli)   -> R2={score_baseline:.3f}  MAE={mae_baseline:.3f}")
        print(f"  TSTR     (train=sintetis, test=asli) -> R2={score_tstr:.3f}  MAE={mae_tstr:.3f}")
        metric_name = "R2"

    gap = score_baseline - score_tstr
    utility_ratio = (score_tstr / score_baseline * 100) if score_baseline != 0 else 0
    print(f"  Selisih performa ({metric_name}): {gap:.3f}  |  Rasio utility: {utility_ratio:.1f}%")

    if utility_ratio >= 90:
        verdict = "SANGAT BAIK - sinyal data sintetis hampir setara data asli"
    elif utility_ratio >= 75:
        verdict = "BAIK - masih layak dipakai untuk kompetisi"
    elif utility_ratio >= 50:
        verdict = "CUKUP - pertimbangkan tuning ulang model sintesis"
    else:
        verdict = "KURANG - data sintetis kemungkinan kehilangan sinyal penting"
    print(f"  Verdict: {verdict}")

    return {
        "metric_name": metric_name,
        "score_baseline": score_baseline,
        "score_tstr": score_tstr,
        "utility_ratio": utility_ratio,
        "verdict": verdict,
    }

In [15]:
# ----------------------------------------------------------------------
# 3. PRIVACY (Distance to Closest Record)
# ----------------------------------------------------------------------

def evaluate_privacy(real: pd.DataFrame, synth: pd.DataFrame, sample_size: int = 1000):
    print("\n=== 3. PRIVACY (Distance to Closest Record) ===")
    numeric_cols, categorical_cols = split_column_types(real)

    if not numeric_cols:
        print("  [DILEWATI] Tidak ada kolom numerik untuk hitung jarak.")
        return None

    r = real[numeric_cols].dropna()
    s = synth[numeric_cols].dropna()

    if len(r) == 0 or len(s) == 0:
        print("  [DILEWATI] Data numerik kosong setelah dropna.")
        return None

    # sample supaya tidak berat kalau datanya besar
    r_sample = r.sample(min(sample_size, len(r)), random_state=42)
    s_sample = s.sample(min(sample_size, len(s)), random_state=42)

    scaler = StandardScaler().fit(r_sample)
    r_scaled = scaler.transform(r_sample)
    s_scaled = scaler.transform(s_sample)

    # jarak euclidean tiap baris sintetis ke baris asli terdekat
    from scipy.spatial import cKDTree
    tree = cKDTree(r_scaled)
    distances, _ = tree.query(s_scaled, k=1)

    min_dist = distances.min()
    mean_dist = distances.mean()
    pct_near_duplicate = (distances < 1e-3).mean() * 100  # hampir identik

    print(f"  Jarak minimum ke data asli terdekat : {min_dist:.4f}")
    print(f"  Jarak rata-rata ke data asli terdekat: {mean_dist:.4f}")
    print(f"  Persentase baris sintetis yang HAMPIR IDENTIK dengan data asli: {pct_near_duplicate:.2f}%")

    if pct_near_duplicate > 5:
        verdict = "RISIKO - banyak baris sintetis terlalu mirip data asli, sebaiknya tambah noise/regularisasi model"
    elif pct_near_duplicate > 1:
        verdict = "PERLU DICEK - ada beberapa baris yang sangat mirip data asli"
    else:
        verdict = "AMAN - baris sintetis cukup berbeda dari data asli"
    print(f"  Verdict: {verdict}")

    return {
        "min_dist": min_dist,
        "mean_dist": mean_dist,
        "pct_near_duplicate": pct_near_duplicate,
        "verdict": verdict,
    }

In [16]:
# ----------------------------------------------------------------------
# 4. VISUAL (PCA 2D)
# ----------------------------------------------------------------------

def evaluate_visual(real: pd.DataFrame, synth: pd.DataFrame, outdir: Path):
    print("\n=== 4. VISUAL CHECK (PCA 2D) ===")
    numeric_cols, _ = split_column_types(real)
    if len(numeric_cols) < 2:
        print("  [DILEWATI] Butuh minimal 2 kolom numerik untuk PCA.")
        return

    r = real[numeric_cols].dropna()
    s = synth[numeric_cols].dropna()

    scaler = StandardScaler().fit(r)
    r_scaled = scaler.transform(r)
    s_scaled = scaler.transform(s)

    pca = PCA(n_components=2, random_state=42).fit(r_scaled)
    r_pca = pca.transform(r_scaled)
    s_pca = pca.transform(s_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].scatter(r_pca[:, 0], r_pca[:, 1], alpha=0.4, label="Asli", color="steelblue", s=15)
    axes[0].set_title("Sebaran Data (PCA 2D): Asli")
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[0].legend()
    
    axes[1].scatter(s_pca[:, 0], s_pca[:, 1], alpha=0.4, label="Sintetis", color="orangered", s=15)
    axes[1].set_title("Sebaran Data (PCA 2D): Sintetis")
    axes[1].set_xlabel("PC1")
    axes[1].set_ylabel("PC2")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(outdir / "pca_asli_vs_sintetis.png", dpi=120)
    plt.close()
    print(f"  -> Plot PCA disimpan: {outdir / 'pca_asli_vs_sintetis.png'}")
    print("  Cek manual: Bandingkan sebaran (biru vs oranye) pada dua grafik yang terpisah.")

In [17]:
# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Evaluasi kualitas data sintetis tabular")
    parser.add_argument("--real", default="medical.csv", help="Path CSV data asli")
    parser.add_argument("--synthetic", default="medical.csv", help="Path CSV data sintetis")
    parser.add_argument("--target", default=None, help="Nama kolom target (opsional, untuk uji utility)")
    parser.add_argument("--task", choices=["classification", "regression"], default="classification",
                         help="Jenis task untuk uji utility (default: classification)")
    parser.add_argument("--outdir", default="hasil_evaluasi", help="Folder output untuk plot & laporan")
    args, _ = parser.parse_known_args()

    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    print(f"Membaca data asli     : {args.real}")
    real = pd.read_csv(args.real)
    
    # Jika path file sintetis sama dengan file asli, kita asumsikan user ingin membuat data sintetis baru
    if args.synthetic == args.real:
        print(f"[INFO] Path data asli dan sintetis sama ({args.real}).")
        print("Sistem akan otomatis me-generate data sintetis baru menggunakan algoritma CTGAN...")
        synth = generate_synthetic_data(real, epochs=50)  # Bisa diperbesar epochs-nya (misal 100/300) jika hasil kurang bagus
    else:
        print(f"Membaca data sintetis : {args.synthetic}")
        synth = pd.read_csv(args.synthetic)
        
    real, synth = align_columns(real, synth)

    print(f"\nJumlah baris - asli: {len(real)}, sintetis: {len(synth)}")
    print(f"Jumlah kolom yang dibandingkan: {len(real.columns)}")

    report_lines = []
    report_lines.append("LAPORAN EVALUASI DATA SINTETIS")
    report_lines.append("=" * 50)

    fidelity_score, fidelity_detail = evaluate_fidelity(real, synth, outdir)
    report_lines.append(f"\n1. FIDELITY: skor rata-rata = {fidelity_score:.1f} / 100")
    report_lines.extend(fidelity_detail)

    utility_result = None
    if args.target:
        utility_result = evaluate_utility(real, synth, args.target, args.task)
        if utility_result:
            report_lines.append(
                f"\n2. UTILITY: {utility_result['metric_name']} baseline={utility_result['score_baseline']:.3f}, "
                f"TSTR={utility_result['score_tstr']:.3f}, rasio={utility_result['utility_ratio']:.1f}% "
                f"-> {utility_result['verdict']}"
            )
    else:
        print("\n=== 2. UTILITY ===\n  [DILEWATI] --target tidak diisi.")
        report_lines.append("\n2. UTILITY: dilewati (--target tidak diisi)")

    privacy_result = evaluate_privacy(real, synth)
    if privacy_result:
        report_lines.append(
            f"\n3. PRIVACY: jarak_min={privacy_result['min_dist']:.4f}, "
            f"persen_hampir_identik={privacy_result['pct_near_duplicate']:.2f}% "
            f"-> {privacy_result['verdict']}"
        )

    evaluate_visual(real, synth, outdir)
    report_lines.append(f"\n4. VISUAL: lihat file PNG di folder '{outdir}'")

    # --- Menyimpan Data Sintetis yang sudah selaras ---
    synth_save_path = outdir / "data_sintesis_tersimpan.csv"
    synth.to_csv(synth_save_path, index=False)
    print(f"\n  -> Data hasil sintesis disimpan di: {synth_save_path}")
    report_lines.append(f"\n5. PENYIMPANAN: Data sintesis disimpan di {synth_save_path}")

    # --- Ringkasan akhir ---
    print("\n" + "=" * 50)
    print("RINGKASAN AKHIR")
    print("=" * 50)
    print(f"  Fidelity score : {fidelity_score:.1f} / 100")
    if utility_result:
        print(f"  Utility ratio  : {utility_result['utility_ratio']:.1f}%  ({utility_result['verdict']})")
    if privacy_result:
        print(f"  Privacy risk   : {privacy_result['pct_near_duplicate']:.2f}% hampir identik  ({privacy_result['verdict']})")

    lulus = fidelity_score >= 70
    if utility_result:
        lulus = lulus and utility_result["utility_ratio"] >= 70
    if privacy_result:
        lulus = lulus and privacy_result["pct_near_duplicate"] <= 5

    kesimpulan = "LAYAK dipakai untuk kompetisi" if lulus else "PERLU PERBAIKAN sebelum dipakai untuk kompetisi"
    print(f"\n  KESIMPULAN: {kesimpulan}")
    report_lines.append(f"\nKESIMPULAN AKHIR: {kesimpulan}")

    report_path = outdir / "laporan_evaluasi.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    print(f"\nLaporan lengkap disimpan di: {report_path}")


if __name__ == "__main__":
    main()

Membaca data asli     : medical.csv
[INFO] Path data asli dan sintetis sama (medical.csv).
Sistem akan otomatis me-generate data sintetis baru menggunakan algoritma CTGAN...

=== 0. GENERATE SYNTHETIC DATA (GMM) ===
  Melatih model Gaussian Mixture (GMM) dengan 10 komponen...
  Men-generate 1338 baris data sintetis...
  [SELESAI] Data sintetis berhasil dibuat!

Jumlah baris - asli: 1338, sintetis: 1338
Jumlah kolom yang dibandingkan: 7

=== 1. FIDELITY (kemiripan statistik) ===
  Skor fidelity rata-rata: 97.4 / 100
  [Numerik] age                       KS-stat=0.037  p-value=0.308  skor_mirip=96.3
  [Numerik] bmi                       KS-stat=0.026  p-value=0.750  skor_mirip=97.4
  [Numerik] children                  KS-stat=0.042  p-value=0.192  skor_mirip=95.8
  [Numerik] charges                   KS-stat=0.052  p-value=0.057  skor_mirip=94.8
  [Kategorikal] sex                    chi2-stat_terskala  p-value=0.053  skor_mirip=99.7
  [Kategorikal] smoker                 chi2-stat_ters